### Tyring to implement chromaDB on a large netflix movies dataset


### Step:1 Load the dataset

In [2]:
import pandas as pd

df = pd.read_csv("netflix_titles.csv")

print(df.columns)
print(df.shape)
df.head()

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')
(8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


### Step:2 Cleaning the dataset

In [4]:
df = df.dropna(subset=['description'])
df = df.drop_duplicates(subset=['title']) 
df = df.reset_index(drop=True)

# Optional: cap dataset size while testing
#df = df.head(2000)  # remove/increase once it works end-to-end

### Step:3 Build the ChromaDB Collection

In [5]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="./chroma_db")

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="netflix_movies",
    embedding_function=embed_fn
)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9437.34it/s]


### Step:4 Insert in batches (important for bigger datasets)

### Chroma can Choke if you try to add tens of thousands of rows in one call, so better to batch it.

In [6]:
batch_size = 100

for start in range(0, len(df), batch_size):
    end = start + batch_size
    batch = df.iloc[start:end]

    documents = batch["description"].astype(str).tolist()
    ids = batch["show_id"].astype(str).tolist()

    metadatas = []

    for _, row in batch.iterrows():
        metadatas.append({
            "title": row["title"],
            "release_year": int(row["release_year"]),
            "type": row["type"],
            "director": "" if pd.isna(row["director"]) else row["director"],
            "cast": "" if pd.isna(row["cast"]) else row["cast"],
            "country": "" if pd.isna(row["country"]) else row["country"],
            "rating": row["rating"],
            "duration": row["duration"],
            "listed_in": row["listed_in"]
        })

    collection.add(
        documents=documents,
        ids=ids,
        metadatas=metadatas
    )

    print(f"Inserted rows {start}-{min(end, len(df))}")

print("Total in collection:", collection.count())

Inserted rows 0-100
Inserted rows 100-200
Inserted rows 200-300
Inserted rows 300-400
Inserted rows 400-500
Inserted rows 500-600
Inserted rows 600-700
Inserted rows 700-800
Inserted rows 800-900
Inserted rows 900-1000
Inserted rows 1000-1100
Inserted rows 1100-1200
Inserted rows 1200-1300
Inserted rows 1300-1400
Inserted rows 1400-1500
Inserted rows 1500-1600
Inserted rows 1600-1700
Inserted rows 1700-1800
Inserted rows 1800-1900
Inserted rows 1900-2000
Inserted rows 2000-2100
Inserted rows 2100-2200
Inserted rows 2200-2300
Inserted rows 2300-2400
Inserted rows 2400-2500
Inserted rows 2500-2600
Inserted rows 2600-2700
Inserted rows 2700-2800
Inserted rows 2800-2900
Inserted rows 2900-3000
Inserted rows 3000-3100
Inserted rows 3100-3200
Inserted rows 3200-3300
Inserted rows 3300-3400
Inserted rows 3400-3500
Inserted rows 3500-3600
Inserted rows 3600-3700
Inserted rows 3700-3800
Inserted rows 3800-3900
Inserted rows 3900-4000
Inserted rows 4000-4100
Inserted rows 4100-4200
Inserted rows

### Step:5. Simple similarity search function

In [7]:
def search_movies(query, n_results=5):
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    for i, result in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        print(f"Result {i + 1}:")
        print(f"Title: {metadata['title']}")
        print(f"Description: {result}")
        print(f"Release Year: {metadata['release_year']}")
        print(f"Type: {metadata['type']}")
        print(f"Director: {metadata['director']}")
        print(f"Cast: {metadata['cast']}")
        print(f"Country: {metadata['country']}")
        print(f"Rating: {metadata['rating']}")
        print(f"Duration: {metadata['duration']}")
        print(f"Listed In: {metadata['listed_in']}")
        print("-" * 50)

In [8]:
# Try it
search_movies("Best indian movies with high duration")

Result 1:
Title: Lust Stories
Description: In the companion to 2013's "Bombay Talkies," four short films by four of India's biggest directors explore love, sex and relationships in modern India.
Release Year: 2018
Type: Movie
Director: Zoya Akhtar, Karan Johar, Anurag Kashyap, Dibakar Banerjee
Cast: Vicky Kaushal, Bhumi Pednekar, Radhika Apte, Neha Dhupia, Manisha Koirala, Akash Thosar, Randeep Jha, Neil Bhoopalam, Jaideep Ahlawat, Sanjay Kapoor, Kiara Advani
Country: India
Rating: TV-MA
Duration: 121 min
Listed In: Comedies, Dramas, International Movies
--------------------------------------------------
Result 2:
Title: An American in Madras
Description: Extensive film clips and interviews tell the story of American filmmaker Ellis R. Dungan, who spent 15 years in India and helped define Tamil cinema.
Release Year: 2013
Type: Movie
Director: Karan Bali
Cast: 
Country: India
Rating: TV-PG
Duration: 80 min
Listed In: Documentaries, International Movies
----------------------------------